# Role Drift Env — Kaggle GPU Smoke Test (20 episodes)

**Goal:** Verify Qwen 1.5B GRPO training works end-to-end on GPU before the main 200-episode run.

**Checks:**
1. Reward trends up (not flat/down)
2. KL stays under 0.5 in first 20 episodes
3. Memory and wall-time per episode are acceptable

**Before running:** Set notebook accelerator to **GPU T4x2** or better.

**Estimated time:** 60–120 minutes for 20 episodes on T4.

In [ ]:
import os, shutil, subprocess, sys

# Copy project from read-only input to writable working dir
src = "/kaggle/input/role-drift-env"
dst = "/kaggle/working/role-drift-env"
if not os.path.exists(dst):
    # Your dataset may be zipped or unzipped — handle both
    if os.path.exists(f"{src}/role_drift_env.zip"):
        subprocess.run(["unzip", "-q", f"{src}/role_drift_env.zip", "-d", dst], check=True)
    else:
        shutil.copytree(src, dst)

os.chdir(dst)
print("cwd:", os.getcwd())
print("contents:", os.listdir("."))

# Install package + deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "accelerate", "sentence-transformers",
                "langdetect", "fasttext"], check=True)
print("install done")

In [ ]:
# Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM:', torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 'N/A', 'GB')

In [ ]:
# Add repo root to sys.path (needed for scripts/inspect_transcripts.py)
import sys
sys.path.insert(0, '/kaggle/working/role-drift-env')

# Run 20-episode dry run
import json
import time
from training.train_grpo import GRPOTrainer

scenario_ids = []
with open('data/scenarios/train.jsonl', 'r') as f:
    for line in f:
        scenario_ids.append(json.loads(line)['scenario_id'])

print(f'Loaded {len(scenario_ids)} scenario IDs')
print('First 3:', scenario_ids[:3])

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT = '/kaggle/working/grpo_dryrun_20ep'

trainer = GRPOTrainer(
    model_name=MODEL,
    sft_checkpoint=MODEL,
    output_dir=OUTPUT,
    group_size=4,
    kl_coef=0.05,
    lr=5e-6,
    max_new_tokens=40,
)

start = time.time()
trainer.train(
    scenario_ids,
    num_episodes=20,
    log_every=1,
    save_transcripts_every=5,
    transcript_dir='/kaggle/working/transcripts',
)
elapsed = time.time() - start
print(f'\n20 episodes in {elapsed/60:.1f} min ({elapsed/20:.1f}s per episode)')

In [ ]:
# Analyze results
import json
from pathlib import Path

log_path = Path(OUTPUT) / 'episode_log.jsonl'
with open(log_path, 'r') as f:
    entries = [json.loads(line) for line in f]

returns = [e['mean_return'] for e in entries]
kls = [e['avg_kl'] for e in entries]

print('Returns:', [f'{r:.2f}' for r in returns])
print('KLs:    ', [f'{k:.3f}' for k in kls])
print(f'Max KL: {max(kls):.3f}')
print(f'Reward delta (last - first): {returns[-1] - returns[0]:.2f}')

if max(kls) > 0.5:
    print('\nWARNING: KL > 0.5 — consider lowering LR before main run')
else:
    print('\nKL OK (< 0.5)')

if returns[-1] > returns[0]:
    print('Reward trending UP — good signal')
else:
    print('Reward flat or DOWN — check reward shaping')

# Extrapolate
est_total = elapsed * 10  # 200 / 20 = 10x
print(f'Estimated 200-episode wall time: {est_total/3600:.1f}h')

In [ ]:
# Inspect a few transcripts for reward hacking
from scripts.inspect_transcripts import inspect_transcript
from pathlib import Path

transcript_dir = Path('/kaggle/working/transcripts')
files = sorted(transcript_dir.glob('*.json'))[:5]
for f in files:
    result = inspect_transcript(f)
    print(f"{result['file']} | {result['num_turns']} turns | return={result['episode_return']:.2f}")
    print(f"  Samples: {result['sample_turns']}")
    if result['flags']:
        for flag in result['flags']:
            print(f"  WARNING: {flag}")
    else:
        print('  OK: No suspicious patterns')
    print()

In [ ]:
# Zip outputs for download
import shutil
from pathlib import Path

shutil.make_archive('/kaggle/working/grpo_dryrun_outputs', 'zip', '/kaggle/working')
print('Download: /kaggle/working/grpo_dryrun_outputs.zip')